# WeldFatigueChecker · 真·120+40 全量合并训练（ModelScope 魔搭）

> 目标：在 **源1~4 原始 120 数据 + 40 张自采 raw_mine** 上做全量合并训练，
> 权重热启动自 120 基础版（last.pt），产出能同时覆盖「120 域 + 自采 40 图」的模型。

## ⚠️ 前置：必须集齐 5 份数据，缺一不可
| 数据 | 获取方式 | 本机是否已有 |
|------|---------|------------|
| last.pt（120 权重） | `C:\\Users\\Administrator\\Desktop\\log\\last.pt` | ✅ 有 |
| 源1 raw（huangyebiaoke, 810MB） | `Desktop/log/steel-tube-dataset-all.zip` 上传解压，或脚本 ghproxy 自动下 | ✅ 有 zip |
| 源2 raw_jian（Roboflow） | 重开同一 PAI-DSW 实例应已在持久化存储（120 训练时上传过）；缺失才需代理重传 `raw_jian/` | ❌ 墙（403） |
| 源3 raw_lohi（Google Drive） | 重开同一 PAI-DSW 实例应已在持久化存储；缺失才需代理 gdown 拉取 `raw_lohi/` | ❌ 墙（502） |
| 源4 raw_kunkun（Roboflow） | 重开同一 PAI-DSW 实例应已在持久化存储；缺失才需代理重传 `raw_kunkun/` | ❌ 墙（403） |
| 自采 raw_mine（40 张） | 本机 `ml/raw_mine/`（被 gitignore，需手动传） | ✅ 有 |

**源2~4 在 120 训练时已手动上传进 PAI-DSW 实例，且 PAI-DSW 关实例不清数据（/mnt/workspace 持久化），重开同一实例应仍在。**
**仅当确实缺失时，才需在魔搭终端 `export HTTPS_PROXY=...` 配代理后从 Roboflow/GDrive 导出重传；不要一上来就忙代理。**
**若缺任意一份，下方「预检」cell 会直接终止，绝不会退回 40-only 塌方。**


In [ ]:
!pip install -q ultralytics pyyaml coremltools
import torch, ultralytics
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| ultralytics', ultralytics.__version__)

In [ ]:
import os, shutil
print('cwd:', os.getcwd())
os.makedirs('ml', exist_ok=True)
%cd ml
CKPT = '../last.pt'
# 源1：若 raw/ 无图片，则把本机 810MB zip 就位并解压到 raw/
IMG_EXTS = ('.jpg','.jpeg','.png','.bmp','.JPG','.PNG')
has_raw = os.path.isdir('raw') and any(f.lower().endswith(IMG_EXTS) for f in os.listdir('raw'))
if not has_raw:
    z_src = '../steel-tube-dataset-all.zip'
    if os.path.isfile(z_src):
        os.makedirs('raw', exist_ok=True)
        shutil.copy(z_src, 'raw/steel-tube-dataset-all.zip')
        !unzip -q -o raw/steel-tube-dataset-all.zip -d raw
        print('源1 解压完成 -> raw/')
    else:
        print('⚠️ 未找到 ../steel-tube-dataset-all.zip，将由脚本 ghproxy 自动下载（需 CN 云可达 GitHub）')
# 可选：若你用旧 ml_kaggle_upload.zip（含 raw_mine）上传，可解压
if os.path.isfile('../ml_kaggle_upload.zip'):
    !unzip -q -o ../ml_kaggle_upload.zip -d .
print('120 权重存在 :', os.path.isfile(CKPT))
print('ml 内容 :', sorted(os.listdir('.'))[:30])

In [ ]:
import os
REQUIRED = {
    'raw':       '源1 huangyebiaoke（810MB，解压自 steel-tube-dataset-all.zip）',
    'raw_jian':  '源2 JIAN SONG（重开同一 PAI-DSW 实例应已在持久化存储；缺失才需代理/重传）',
    'raw_lohi':  '源3 LoHi-WELD（同上，应在持久化存储；缺失才需代理 gdown/重传）',
    'raw_kunkun':'源4 kunkun-vhmx2/weld（同上，应在持久化存储；缺失才需代理/重传）',
    'raw_mine':  '自采 40 张（本机 ml/raw_mine，需上传到实例）',
}
IMG_EXTS = ('.jpg','.jpeg','.png','.bmp','.JPG','.PNG')
print('==== 120+40 预检 ====')
missing = []
for d, desc in REQUIRED.items():
    ok = os.path.isdir(d) and any(f.lower().endswith(IMG_EXTS) for f in os.listdir(d))
    print(('  ✓ ' if ok else '  ✗ ') + d + ('  ' + desc if not ok else ''))
    if not ok:
        missing.append(d)
if missing:
    print('\n❌ 预检未通过，禁止训练（否则会退回 40-only 塌方）：')
    print('   缺失:', missing)
    print('   请按上方说明补齐数据后重跑本 cell，再继续训练 cell。')
    raise SystemExit(1)
print('\n✅ 5 份数据齐全，可进行真·120+40 全量合并训练。')

In [ ]:
# 真·120+40：用 120 权重热启动，在「源1~4 + raw_mine」全量合并集上训 120 轮
# 注意：不加 --finetune（那只训40张会塌方）、不加 --incremental（要全量重训不是只训新增）
!python weld_train.py --init-from ../last.pt --epochs 120

In [ ]:
# 导出验证关卡：weld_train.py 自动导出的 CoreML 必须是真 5 类（防止再出 80 类 COCO 废模型）
from coremltools.proto import Model_pb2
import glob, os
pkg = glob.glob('ml/WeldDefectModel.mlpackage/Data/com.apple.CoreML/model.mlmodel') or \
      glob.glob('WeldDefectModel.mlpackage/Data/com.apple.CoreML/model.mlmodel')
assert pkg, '未找到导出的 WeldDefectModel.mlpackage'
spec = Model_pb2.Model()
spec.ParseFromString(open(pkg[0], 'rb').read())
nms = spec.pipeline.models[1].nonMaximumSuppression
labels = list(nms.stringClassLabels.vector)
EXPECT = ['porosity', 'crack', 'undercut', 'overlap', 'unfused']
assert labels == EXPECT, f'❌ 导出的不是 5 类模型（检查 weld_train.py 是否从 best.pt 导出）: 前8个={labels[:8]}'
shapes = {i.name: list(i.type.multiArrayType.shape)
          for i in spec.pipeline.models[1].description.input
          if i.type.WhichOneof('Type') == 'multiArrayType'}
assert any(s[-1] == 5 for s in shapes.values()), f'❌ 分数张量不是 5 类: {shapes}'
nms.confidenceThreshold = 0.05
open(pkg[0], 'wb').write(spec.SerializeToString())
print('✅ 120+40 导出验证通过：5 类 | 分数形状', shapes, '| NMS 阈值 0.05 → 可下载替换')

In [ ]:
import glob, os
runs = sorted(glob.glob('runs/detect/weld_defect_5cls*/weights/best.pt'))
print('最新训练产物 :', runs[-1] if runs else '未找到')
print('mlpackage 已导出 :', os.path.isdir('WeldDefectModel.mlpackage'))
print('\n==== 部署前三道门（任一不过禁止替换）====')
print('① 合并集 mAP50 ≥ max(0.93, 0.983-0.03) = 0.953')
print('② 自采域（那 40 张）检出率 ≥ 80%')
print('③ 五类（porosity/crack/undercut/overlap/unfused）无归零')
print('验证通过后，再下载 mlpackage 回仓库替换。')

## 收尾：拿回仓库 → git 推送
1. 左侧文件面板找到 `ml/WeldDefectModel.mlpackage` 右键下载。
2. 覆盖到仓库的 `WeldDefectModel.mlpackage`。
3. 本地 Git Bash：
   ```
   git add WeldDefectModel.mlpackage
   git commit -m "fix(model): 真·120+40 全量合并训练权重"
   git push origin main
   ```
4. 等 Actions 变绿 → 下载 IPA → 侧载（底栏 `BUILD:` 应为本次 sha）→ 复测那张焊瘤图。
